<a href="https://colab.research.google.com/github/HadyShibli/ter_1/blob/main/CloudTut4_final_exercise.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import requests
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import ipywidgets as widgets
from ipywidgets import VBox, HBox, Output
from IPython.display import display, HTML

pd.set_option("display.max_columns", None)

In [3]:
# The API returns max 32,000 rows per request
RESOURCE_ID = "053cea08-09bc-40ec-8f7a-156f0677aff3"
URL = "https://data.gov.il/api/3/action/datastore_search"
LIMIT = 32000

response = requests.get(URL, params={"resource_id": RESOURCE_ID, "limit": LIMIT}, timeout=120)
data = response.json()
data_df = pd.DataFrame(data["result"]["records"])

# Numeric columns -> numbers
for col in ["shnat_yitzur", "tozeret_cd", "degem_cd", "ramat_eivzur_betihuty",
            "kvutzat_zihum", "tzeva_cd", "horaat_rishum", "mispar_rechev"]:
    if col in data_df.columns:
        data_df[col] = pd.to_numeric(data_df[col], errors="coerce")

# Text columns -> clean strings
for col in ["tozeret_nm", "kinuy_mishari", "ramat_gimur"]:
    data_df[col] = data_df[col].fillna("").astype(str).str.strip()

print("Records loaded:", len(data_df))
data_df.head()

Records loaded: 32000


,_id,mispar_rechev,tozeret_cd,sug_degem,tozeret_nm,degem_cd,degem_nm,ramat_gimur,ramat_eivzur_betihuty,kvutzat_zihum,shnat_yitzur,degem_manoa,mivchan_acharon_dt,tokef_dt,baalut,misgeret,tzeva_cd,tzeva_rechev,zmig_kidmi,zmig_ahori,sug_delek_nm,horaat_rishum,moed_aliya_lakvish,kinuy_mishari
0,1,70540001,885,P,קיה ד. קוריאה,488,DC511G,GX,1.0,13.0,2019,G3LC,2026-11-08,2027-10-26,פרטי,KNADC511GL6336063,10,שחור,185/65R15,185/65R15,בנזין,190019.0,2019-10,RIO
1,2,4136652,885,P,קיה ד. קוריאה,37,BX511A,EX,NaN,6.0,2013,G3LA,2026-11-08,2027-10-13,פרטי,KNABX511AET643200,93,כסוף כהה,165/60R14,165/60R14,בנזין,130171.0,2013-10,PICANTO
2,3,2716132,751,P,שברולט ד.קוריא,73,CD6E1,LS,NaN,14.0,2014,B10D1,2026-11-08,2027-06-28,פרטי,KL1CD6E19EC550154,80,שנהב לבן,155/80R13,155/80R13,בנזין,140308.0,2014-6,SPARK
3,4,84140401,845,P,יונדאי טורקיה,64,B351A,PRIME,NaN,11.0,2019,G3LC,2026-11-08,2027-11-10,פרטי,NLHB351AGLZ593429,80,שנהב לבן,185/65R15,185/65R15,בנזין,180221.0,2019-11,I20
4,5,53738203,481,P,יונדאי ד. קורי,267,L54JJ,LIMITED,6.0,4.0,2023,G4NR,2026-11-08,2027-09-09,פרטי,KMHL54JJ6RA072465,11,שחור מטלי,215/55R17,215/55R17,בנזין,200881.0,2023-9,SONATA HYBRID


In [4]:
def show_plotly(fig):
    display(HTML(fig.to_html(include_plotlyjs="cdn", full_html=False)))

In [5]:
tab1_content = Output()
with tab1_content:
    print("Data Overview:")
    display(data_df.describe())
    print("\nText columns:")
    display(data_df.describe(include="object"))

In [6]:
PAGE_SIZE = 50
n_pages = (len(data_df) - 1) // PAGE_SIZE + 1

page_slider = widgets.IntSlider(value=1, min=1, max=n_pages, description="Page:",
                                continuous_update=False, layout=widgets.Layout(width="500px"))
search_box = widgets.Text(placeholder="Filter by manufacturer / model...", description="Search:")
table_out = Output()

def get_filtered_raw():
    text = search_box.value.strip()
    if not text:
        return data_df
    mask = (data_df["tozeret_nm"].str.contains(text, case=False, na=False) |
            data_df["kinuy_mishari"].str.contains(text, case=False, na=False))
    return data_df[mask]

def update_table(change=None):
    df = get_filtered_raw()
    pages = max(1, (len(df) - 1) // PAGE_SIZE + 1)
    page_slider.max = pages
    start = (page_slider.value - 1) * PAGE_SIZE
    table_out.clear_output()
    with table_out:
        print(f"Showing rows {start + 1}-{min(start + PAGE_SIZE, len(df))} of {len(df)}")
        display(df.iloc[start:start + PAGE_SIZE])

def on_search(change):
    page_slider.value = 1
    update_table()

page_slider.observe(update_table, names="value")
search_box.observe(on_search, names="value")
update_table()

tab2_content = Output()
with tab2_content:
    print("Raw Data:")
    display(VBox([HBox([search_box, page_slider]), table_out]))

In [7]:
year_mode = widgets.ToggleButtons(options=["Number of vehicles", "Number of different models"],
                                  description="Show:")
year_out = Output()

def update_year_chart(change=None):
    year_out.clear_output()
    years_df = data_df.dropna(subset=["shnat_yitzur"])
    if year_mode.value == "Number of vehicles":
        counts = years_df.groupby("shnat_yitzur").size()
    else:
        counts = years_df.groupby("shnat_yitzur")["kinuy_mishari"].nunique()
    counts.index = counts.index.astype(int)
    with year_out:
        fig, axes = plt.subplots(1, 2, figsize=(16, 5))
        axes[0].bar(counts.index, counts.values, color="#8a5082")
        axes[0].set_title(f"{year_mode.value} per production year")
        axes[0].set_xlabel("shnat_yitzur")
        axes[0].set_ylabel(year_mode.value)
        axes[0].grid(True, axis="y", alpha=0.3)
        sns.histplot(years_df["shnat_yitzur"], bins=10, kde=True, color="blue", ax=axes[1])
        axes[1].set_title("Distribution of Age")
        plt.tight_layout()
        plt.show()

year_mode.observe(update_year_chart, names="value")
update_year_chart()

tab3_content = Output()
with tab3_content:
    display(VBox([year_mode, year_out]))

In [8]:
tab4_content = Output()
with tab4_content:
    top_makers = data_df["tozeret_nm"].value_counts().head(15).index
    heat = (data_df[data_df["tozeret_nm"].isin(top_makers)]
            .dropna(subset=["shnat_yitzur"])
            .pivot_table(index="tozeret_nm", columns="shnat_yitzur",
                         values="mispar_rechev", aggfunc="count", fill_value=0))
    heat.columns = heat.columns.astype(int)
    plt.figure(figsize=(16, 7))
    sns.heatmap(heat, cmap="viridis", cbar_kws={"label": "Number of vehicles"})
    plt.title("Vehicles per year - top 15 manufacturers")
    plt.xlabel("Production year")
    plt.ylabel("Manufacturer")
    plt.tight_layout()
    plt.show()

In [9]:
tab5_content = Output()
with tab5_content:
    top10 = data_df["tozeret_nm"].value_counts().head(10).index
    sun_df = (data_df[data_df["tozeret_nm"].isin(top10)]
              .groupby(["tozeret_nm", "kinuy_mishari"]).size()
              .reset_index(name="count"))
    sun_df["kinuy_mishari"] = sun_df["kinuy_mishari"].replace("", "(unknown)")
    fig = px.sunburst(sun_df, path=["tozeret_nm", "kinuy_mishari"], values="count",
                      title="Top 10 manufacturers and their models")
    fig.update_layout(height=650)
    show_plotly(fig)

In [10]:
tozeret_dropdown = widgets.Dropdown(
    options=[""] + sorted(data_df["tozeret_nm"].replace("", np.nan).dropna().unique().tolist()),
    description="Tozeret:", style={"description_width": "initial"})
kinuy_dropdown = widgets.Dropdown(options=[""], description="Kinuy Mishari:",
                                  style={"description_width": "initial"})
gimur_dropdown = widgets.Dropdown(options=["All"], description="Ramat Gimur:",
                                  style={"description_width": "initial"})
search_out = Output()

def filtered_by_model():
    return data_df[(data_df["tozeret_nm"] == tozeret_dropdown.value) &
                   (data_df["kinuy_mishari"] == kinuy_dropdown.value)]

def update_kinuy(change):
    if change["new"]:
        models = data_df[data_df["tozeret_nm"] == change["new"]]["kinuy_mishari"]
        kinuy_dropdown.options = [""] + sorted(models[models != ""].unique().tolist())
    else:
        kinuy_dropdown.options = [""]

def update_gimur(change):
    if tozeret_dropdown.value and kinuy_dropdown.value:
        g = filtered_by_model()["ramat_gimur"]
        gimur_dropdown.options = ["All"] + sorted(g[g != ""].unique().tolist())
    else:
        gimur_dropdown.options = ["All"]
    update_search_output()

def update_search_output(change=None):
    search_out.clear_output()
    with search_out:
        if not (tozeret_dropdown.value and kinuy_dropdown.value):
            print("Please select a manufacturer and a model.")
            return
        df = filtered_by_model()
        gimurs = sorted(df["ramat_gimur"][df["ramat_gimur"] != ""].unique().tolist())
        if gimur_dropdown.value != "All":
            df = df[df["ramat_gimur"] == gimur_dropdown.value]
        print(f"Total Records: {len(df)}")
        print(f"Unique Ramat Gimur: {', '.join(gimurs) if gimurs else 'None'}")
        if gimur_dropdown.value != "All":
            print(f"Selected Ramat Gimur: {gimur_dropdown.value}")
        display(df.head(20))

tozeret_dropdown.observe(update_kinuy, names="value")
kinuy_dropdown.observe(update_gimur, names="value")
gimur_dropdown.observe(update_search_output, names="value")
update_search_output()

tab6_content = Output()
with tab6_content:
    display(VBox([tozeret_dropdown, kinuy_dropdown, gimur_dropdown, search_out]))

In [11]:
tabs = widgets.Tab(children=[tab1_content, tab2_content, tab3_content,
                             tab4_content, tab5_content, tab6_content])
titles = ["Tab 1: Data Overview", "Tab 2: Raw Data", "Tab 3: Year Count",
          "Tab 4: Heatmap", "Tab 5: Sunburst", "Tab 6: Search"]
for i, t in enumerate(titles):
    tabs.set_title(i, t)

display(tabs)